# Multilingual Emotion Analysis with BERT\n\nStarter notebook using `bert-base-multilingual-cased` for Korean, Chinese, and English emotion classification.

In [ ]:
# Optional: install dependencies if needed\n# !pip install pandas scikit-learn torch transformers datasets

In [ ]:
import pandas as pd\nfrom pathlib import Path\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import LabelEncoder\n\nfrom datasets import Dataset\nfrom transformers import (\n    AutoTokenizer,\n    AutoModelForSequenceClassification,\n    DataCollatorWithPadding,\n    TrainingArguments,\n    Trainer\n)

In [ ]:
DATA_DIR = Path('../data')\nfiles = ['korean.csv', 'chinese.csv', 'english.csv']\n\ndfs = [pd.read_csv(DATA_DIR / f) for f in files]\ndf = pd.concat(dfs, ignore_index=True)\ndf.head()

In [ ]:
label_encoder = LabelEncoder()\ndf['label_id'] = label_encoder.fit_transform(df['label'])\nlabel2id = {label: int(i) for i, label in enumerate(label_encoder.classes_)}\nid2label = {int(i): label for label, i in label2id.items()}\n\ntrain_df, val_df = train_test_split(\n    df,\n    test_size=0.2,\n    random_state=42,\n    stratify=df['label_id'] if df['label_id'].nunique() > 1 else None\n)\n\ntrain_dataset = Dataset.from_pandas(train_df[['text', 'label_id']], preserve_index=False)\nval_dataset = Dataset.from_pandas(val_df[['text', 'label_id']], preserve_index=False)

In [ ]:
model_name = 'bert-base-multilingual-cased'\ntokenizer = AutoTokenizer.from_pretrained(model_name)\n\ndef preprocess(batch):\n    return tokenizer(batch['text'], truncation=True)\n\ntrain_dataset = train_dataset.map(preprocess, batched=True)\nval_dataset = val_dataset.map(preprocess, batched=True)\n\ntrain_dataset = train_dataset.rename_column('label_id', 'labels')\nval_dataset = val_dataset.rename_column('label_id', 'labels')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(\n    model_name,\n    num_labels=len(label2id),\n    id2label=id2label,\n    label2id=label2id\n)\n\ndata_collator = DataCollatorWithPadding(tokenizer=tokenizer)\n\ntraining_args = TrainingArguments(\n    output_dir='../results/checkpoints',\n    learning_rate=2e-5,\n    per_device_train_batch_size=8,\n    per_device_eval_batch_size=8,\n    num_train_epochs=2,\n    eval_strategy='epoch',\n    save_strategy='epoch',\n    logging_dir='../results/logs',\n    logging_steps=10\n)

In [ ]:
trainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=train_dataset,\n    eval_dataset=val_dataset,\n    tokenizer=tokenizer,\n    data_collator=data_collator\n)\n\n# trainer.train()\n# trainer.evaluate()

In [ ]:
# Save model when ready\n# trainer.save_model('../models/bert-multilingual-emotion')\n# tokenizer.save_pretrained('../models/bert-multilingual-emotion')